In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [3]:
#temizlenmiş veriyi okuyup boş satırları siliyoruz
df = pd.read_csv("temizlenmis_veri_seti.csv")
df = df.dropna(subset=['temiz_review'])


In [4]:
#X girdisi yorumlar, y girdisi pozitif negatif gibi sınıflar
X = df['temiz_review']
y = df['score']

In [5]:
#veri %80 eğitim ve %20 test olarak rastgele bölüyoruz
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"eğitim için ayrılan satır sayısı: {len(X_train)}")
print(f"test için ayrılan satır sayısı: {len(X_test)}")

eğitim için ayrılan satır sayısı: 35984
test için ayrılan satır sayısı: 8996


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [7]:
#TF-IDF vectorizeri oluşturuyoruz, bigram ikili kelimeleri de dahil edeiyoruz
vectorizer = TfidfVectorizer(ngram_range=(1,2))

In [8]:
#fit ile öğrenip transform ile sayıyoruz normalde bunlar ayrı ayrı
X_train_vektor = vectorizer.fit_transform(X_train)

In [9]:
#test verisini matematiğe çeviriyoruz ama öğrenme yapmıyoruz
X_test_vektor = vectorizer.transform(X_test)

print("kelimeler başarıyla vektörlere(matematiğe) dönüştürüldü")
print(f"toplam öğrenilen kelime/bigram sayısı: {X_train_vektor.shape[1]}")

kelimeler başarıyla vektörlere(matematiğe) dönüştürüldü
toplam öğrenilen kelime/bigram sayısı: 172679


Naive Bayes Modelinin Eğitimi

In [10]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

In [11]:
#modeli oluşturuyoruz
nb_model = MultinomialNB()

In [12]:
#modeli eğitiyoruz
nb_model.fit(X_train_vektor,y_train)

#ona soruları verip tahmin yapmasını istiyoruz
y_tahmin = nb_model.predict(X_test_vektor)

#modelin cevapları ile doğru cevapları karşılaştırıp kıyaslıyoruz
dogruluk = accuracy_score(y_test, y_tahmin)
print(f"Naive Bayes modelinin sınav başarısı: %{dogruluk*100:.2f}\n")

#hangi duyguyu daha iyi bilmiş
print(classification_report(y_test, y_tahmin))

Naive Bayes modelinin sınav başarısı: %76.78

              precision    recall  f1-score   support

    Negative       0.86      0.76      0.81      2960
     Neutral       0.69      0.66      0.68      2956
    Positive       0.76      0.88      0.81      3080

    accuracy                           0.77      8996
   macro avg       0.77      0.77      0.77      8996
weighted avg       0.77      0.77      0.77      8996



Lojistik Regresyon Modelinin Eğitimi


In [13]:
from sklearn.linear_model import LogisticRegression

In [14]:
log_model = LogisticRegression(max_iter=1000)
#max_iter ile daha fazla deneme hakkı veriyoruz, varsayılan 100

In [15]:
#modeli eğitiyoruz
log_model.fit(X_train_vektor, y_train)

#model sınava giriyor
y_tahmin_log = log_model.predict(X_test_vektor)

#sınav sonuçları
dogruluk_log = accuracy_score(y_test, y_tahmin_log)

print(f"lojistik regresyon başarısı: %{dogruluk_log * 100:.2f}\n")

#detaylı karne
print(classification_report(y_test, y_tahmin_log))

lojistik regresyon başarısı: %76.68

              precision    recall  f1-score   support

    Negative       0.83      0.78      0.80      2960
     Neutral       0.69      0.67      0.68      2956
    Positive       0.79      0.85      0.81      3080

    accuracy                           0.77      8996
   macro avg       0.77      0.77      0.77      8996
weighted avg       0.77      0.77      0.77      8996



SVM Modeli Eğitimi


In [16]:
from sklearn.svm import LinearSVC

In [17]:
svm_model = LinearSVC()

In [18]:
#modeli eğitiyoruz
svm_model.fit(X_train_vektor, y_train)

#model sınava giriyor
y_tahmin_svm = svm_model.predict(X_test_vektor)

#sınav sonuçları
dogruluk_svm = accuracy_score(y_test, y_tahmin_svm)

print(f"SVM modelinin başarısı: {dogruluk_svm * 100:.2f}\n")

#detaylı karne
print(classification_report(y_test, y_tahmin_svm))


SVM modelinin başarısı: 76.23

              precision    recall  f1-score   support

    Negative       0.82      0.79      0.80      2960
     Neutral       0.68      0.66      0.67      2956
    Positive       0.79      0.84      0.81      3080

    accuracy                           0.76      8996
   macro avg       0.76      0.76      0.76      8996
weighted avg       0.76      0.76      0.76      8996



En iyi model olan Naive Bayesin kaydedilmesi

In [19]:
#joblib kütüphanesi kullanarak bilgisayara paket halinde kaydedilecektir.
import joblib
import os

In [22]:
#modelleri toplu tutmak için klasör oluşturalım 
os.makedirs("models", exist_ok=True)

#sözlüğümüzü kaydediyoruz gelecek yeni kelimeleri sayıya çevirebilmesi için
joblib.dump(vectorizer, "models/vectorizer.pkl")

#kazanan modeli yani naive bayesi kaydediyoruz (tahmin yapabilmesi için)
joblib.dump(nb_model,"models/nb_model.pkl")

print("modeller başarıyla models klasörüne kaydedildi")

modeller başarıyla models klasörüne kaydedildi
